# EventHub AI Create Event V2 — Pipeline 01: Validate Dataset
### Mục tiêu:
1. Mount Google Drive và kiểm tra cấu trúc thư mục dữ liệu EventHub AI.
2. Đọc dataset gốc hoặc dataset đã làm sạch từ Drive mà **không làm thay đổi file gốc**.
3. Chạy bộ kiểm định chất lượng (Validation Engine): Kiểm tra tính hợp lệ của JSON, Schema V2, ngày giờ, sessions, tickets và phát hiện hallucination.
4. Xuất báo cáo chất lượng tổng quát dưới dạng Markdown và JSON vào Google Drive.



In [ ]:
# [1] Mount Google Drive và thiết lập đường dẫn làm việc
from google.colab import drive
import os
import sys
import json
import zipfile

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/EventHub_AI'
RAW_DATA_DIR = os.path.join(DRIVE_ROOT, 'dataset/raw')
CLEANED_DATA_DIR = os.path.join(DRIVE_ROOT, 'dataset/cleaned/eventhub_v2')
REPORT_DIR = os.path.join(DRIVE_ROOT, 'dataset/reports')

os.makedirs(REPORT_DIR, exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")
print(f"Report dir: {REPORT_DIR}")



In [ ]:
# [2] Kiểm tra và giải nén tệp dataset ZIP nếu cần
zip_path = os.path.join(RAW_DATA_DIR, 'EventHub_AI_Create_Event_Dataset_300.zip')
extract_target = os.path.join(RAW_DATA_DIR, 'EventHub_AI_Create_Event_Dataset_300')

if os.path.exists(zip_path) and not os.path.exists(extract_target):
    print(f"Đang giải nén {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(RAW_DATA_DIR)
    print("Giải nén hoàn tất!")
elif os.path.exists(extract_target):
    print(f"Thư mục dữ liệu đã sẵn sàng: {extract_target}")
else:
    print(f"Lưu ý: Không tìm thấy ZIP tại {zip_path}. Sẽ sử dụng dữ liệu đã làm sạch tại {CLEANED_DATA_DIR}")



In [ ]:
# [3] Định nghĩa bộ kiểm định chất lượng Dataset Validator (Schema V2)
import re
from datetime import datetime

DATE_REGEX = re.compile(r'^\d{4}-\d{2}-\d{2}$')
TIME_REGEX = re.compile(r'^([01]\d|2[0-3]):[0-5]\d$')

VALID_CATEGORIES = {
    'am-nhac-bieu-dien', 'kinh-doanh-cong-nghe', 'hoi-thao-giao-duc',
    'the-thao', 'nghe-thuat-trien-lam', 'am-thuc-do-uong',
    'le-hoi-van-hoa', 'cong-dong-thien-nguyen', 'giai-tri-doi-song', 'other'
}

def validate_eventhub_record(rec, idx=0):
    errors = []
    warnings = []
    sample_id = rec.get("id", f"SAMPLE-{idx}")

    output_json = rec.get("output_json")
    if not isinstance(output_json, dict):
        return [f"Sample {sample_id}: output_json không phải là dict hợp lệ"], []

    title = output_json.get("title")
    if not title or not str(title).strip():
        errors.append(f"Sample {sample_id}: Thiếu title hoặc title rỗng")

    category = output_json.get("category")
    if category and category not in VALID_CATEGORIES:
        warnings.append(f"Sample {sample_id}: category '{category}' không nằm trong enum chuẩn")

    short_desc = output_json.get("short_description")
    if short_desc and len(short_desc) > 300:
        warnings.append(f"Sample {sample_id}: short_description quá dài ({len(short_desc)} ký tự)")

    sessions = output_json.get("sessions")
    if not isinstance(sessions, list) or len(sessions) == 0:
        errors.append(f"Sample {sample_id}: sessions phải là mảng có ít nhất 1 phần tử")
    else:
        for s_idx, s in enumerate(sessions):
            s_date = s.get("start_date")
            if not s_date or not DATE_REGEX.match(str(s_date)):
                errors.append(f"Sample {sample_id} Session {s_idx}: start_date không đúng định dạng YYYY-MM-DD ({s_date})")
            s_time = s.get("start_time")
            if s_time and not TIME_REGEX.match(str(s_time)):
                errors.append(f"Sample {sample_id} Session {s_idx}: start_time không đúng định dạng HH:mm ({s_time})")

    tickets = output_json.get("ticket_types")
    if tickets is not None and not isinstance(tickets, list):
        errors.append(f"Sample {sample_id}: ticket_types phải là mảng hoặc null")
    elif isinstance(tickets, list):
        for t_idx, t in enumerate(tickets):
            if not t.get("name"):
                errors.append(f"Sample {sample_id} Ticket {t_idx}: Thiếu tên vé")
            price = t.get("price")
            if price is not None and (not isinstance(price, (int, float)) or price < 0):
                errors.append(f"Sample {sample_id} Ticket {t_idx}: price không hợp lệ ({price})")

    return errors, warnings



In [ ]:
# [4] Thực thi kiểm định trên Dataset Cleaned V2
import os
import json

DRIVE_ROOT = '/content/drive/MyDrive/EventHub_AI'
if 'CLEANED_DATA_DIR' not in globals():
    CLEANED_DATA_DIR = os.path.join(DRIVE_ROOT, 'dataset/cleaned/eventhub_v2')

target_file = os.path.join(CLEANED_DATA_DIR, 'dataset_all.jsonl')
if not os.path.exists(target_file):
    target_file = '/content/dataset_all.jsonl' # fallback nếu upload thủ công

print(f"Bắt đầu kiểm tra file: {target_file}")
total_samples = 0
all_errors = []
all_warnings = []

if os.path.exists(target_file):
    with open(target_file, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f, 1):
            line = line.strip()
            if line:
                total_samples += 1
                try:
                    rec = json.loads(line)
                    errs, warns = validate_eventhub_record(rec, idx)
                    all_errors.extend(errs)
                    all_warnings.extend(warns)
                except Exception as e:
                    all_errors.append(f"Dòng {idx}: Lỗi cú pháp JSON - {str(e)}")

    print(f"\n--- KẾT QUẢ KIỂM ĐỊNH DATASET ---")
    print(f"Tổng số mẫu: {total_samples}")
    print(f"Tổng số lỗi nghiêm trọng (Errors): {len(all_errors)}")
    print(f"Tổng số cảnh báo (Warnings): {len(all_warnings)}")

    if len(all_errors) == 0:
        print("\n✅ CHÚC MỪNG: Dataset V2 đã vượt qua Quality Gate 100%!")
    else:
        print("\n❌ CẢNH BÁO: Phát hiện lỗi cần khắc phục trước khi huấn luyện:")
        for err in all_errors[:10]:
            print(f"  - {err}")
else:
    print(f"Tệp không tồn tại: {target_file}. Vui lòng kiểm tra lại đường dẫn Drive.")



In [ ]:
# [5] Xuất báo cáo kiểm tra ra Drive
report_md = os.path.join(REPORT_DIR, 'COLAB_VALIDATION_REPORT.md')
with open(report_md, 'w', encoding='utf-8') as f:
    f.write("# BÁO CÁO KIỂM ĐỊNH DỮ LIỆU TRÊN GOOGLE COLAB\n\n")
    f.write(f"- **Tệp kiểm tra:** `{target_file}`\n")
    f.write(f"- **Tổng số mẫu:** {total_samples}\n")
    f.write(f"- **Số lỗi nghiêm trọng:** {len(all_errors)}\n")
    f.write(f"- **Số cảnh báo:** {len(all_warnings)}\n")
    f.write(f"- **Trạng thái Quality Gate:** **{'ĐẠT (PASS)' if len(all_errors) == 0 else 'KHÔNG ĐẠT (FAIL)'}**\n")

print(f"Báo cáo đã được lưu tại: {report_md}")

